# Sonata Model: Advanced Training and Evaluation Notebook

This notebook provides a comprehensive workflow for fine-tuning and evaluating Sonata-like point cloud segmentation models on custom datasets. It covers the entire lifecycle from data preparation to performance analysis and model export.

**Workflow Overview:**
1.  **Data Preparation**: We will generate a synthetic, labeled dataset and apply data augmentation.
2.  **Simulated Training**: A mock training loop will simulate the fine-tuning process, generating loss curves and a fine-tuned model.
3.  **Evaluation**: We will evaluate the "pre-trained" and "fine-tuned" models using confusion matrices and side-by-side 3D visualizations.
4.  **Advanced Techniques**: We will explore cross-validation and hyperparameter tuning concepts.
5.  **Reporting**: An automated report will summarize the key findings.

--- 
***Note***: *Actually training a large transformer model like Sonata is computationally expensive and complex. This notebook **simulates** the training process to provide a complete, interactive, and educational template for a real-world MLOps pipeline.*

In [ ]:
import os
import json
import time
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import VBox, HBox
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold

# --- Configuration ---
CUSTOM_DATA_DIR = Path('./custom_dataset')
MODELS_DIR = Path('./models')
REPORTS_DIR = Path('./reports')

# Create directories
CUSTOM_DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

print("Directories are set up.")

## 1. Data Preparation & Augmentation

First, we'll generate a custom dataset with known ground-truth labels. Our dataset will consist of point clouds containing simple geometric shapes (a ground plane, a sphere, and a cylinder). We will also define and visualize data augmentation functions.

In [ ]:
def generate_synthetic_cloud(num_points_per_shape=2000):
    """Generates a point cloud with a plane, sphere, and cylinder."""
    # Plane
    plane_points = np.random.rand(num_points_per_shape, 3)
    plane_points[:, 2] = 0.05 * np.random.randn(num_points_per_shape)
    plane_labels = np.ones(num_points_per_shape) * 1 # Label 1 for plane
    
    # Sphere
    phi = np.random.uniform(0, 2 * np.pi, num_points_per_shape)
    costheta = np.random.uniform(-1, 1, num_points_per_shape)
    theta = np.arccos(costheta)
    r = 0.5
    x = r * np.sin(theta) * np.cos(phi) + 0.5
    y = r * np.sin(theta) * np.sin(phi) + 0.5
    z = r * np.cos(theta) + 0.7
    sphere_points = np.vstack((x, y, z)).T
    sphere_labels = np.ones(num_points_per_shape) * 2 # Label 2 for sphere
    
    # Cylinder
    radius, height = 0.3, 1.0
    theta = np.random.uniform(0, 2 * np.pi, num_points_per_shape)
    z = np.random.uniform(0, height, num_points_per_shape)
    x = radius * np.cos(theta) - 0.5
    y = radius * np.sin(theta) - 0.5
    cylinder_points = np.vstack((x, y, z)).T
    cylinder_labels = np.ones(num_points_per_shape) * 3 # Label 3 for cylinder
    
    points = np.vstack([plane_points, sphere_points, cylinder_points])
    labels = np.hstack([plane_labels, sphere_labels, cylinder_labels])
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    return pcd, labels.astype(int)

# Generate and save a sample for training/evaluation
pcd_train, labels_train = generate_synthetic_cloud()
o3d.io.write_point_cloud(str(CUSTOM_DATA_DIR / 'train_sample.ply'), pcd_train)
np.save(str(CUSTOM_DATA_DIR / 'train_sample_labels.npy'), labels_train)

print(f"Generated and saved a synthetic point cloud with {len(pcd_train.points)} points.")

# --- Data Augmentation Functions ---
def augment_cloud(pcd):
    """Applies rotation, scaling, and jittering to a point cloud."""
    points = np.asarray(pcd.points)
    
    # Rotation
    angle = np.random.uniform(-np.pi/4, np.pi/4)
    R = np.array([[np.cos(angle), -np.sin(angle), 0],
                  [np.sin(angle), np.cos(angle), 0],
                  [0, 0, 1]])
    points = points @ R.T
    
    # Scaling
    scale = np.random.uniform(0.9, 1.1)
    points *= scale
    
    # Jitter
    jitter = np.random.normal(0, 0.02, points.shape)
    points += jitter
    
    aug_pcd = o3d.geometry.PointCloud()
    aug_pcd.points = o3d.utility.Vector3dVector(points)
    return aug_pcd

# Visualize augmentation
pcd_aug = augment_cloud(pcd_train)
pcd_train.paint_uniform_color([1, 0, 0]) # Original in red
pcd_aug.paint_uniform_color([0, 1, 0])   # Augmented in green
print("Visualizing data augmentation (original in red, augmented in green):")
o3d.visualization.draw_geometries([pcd_train, pcd_aug])

## 2. Model Definition & Training Simulation

Here, we define our segmentation models and simulate the fine-tuning process. 

- **`pre-trained_model`**: This model simulates a generic, pre-trained Sonata model. It will make some mistakes on our specific dataset.
- **`fine-tuned_model`**: This model simulates the result of fine-tuning. It will be more accurate on our custom geometric shapes.

Click the **'Start Simulated Training'** button to run the mock training loop, which will generate loss curves and produce the fine-tuned model.

In [ ]:
# --- Model Simulation ---
def pretrained_model_segment(pcd):
    """Simulates a pre-trained model with some errors."""
    _, true_labels = generate_synthetic_cloud() # Get the ground truth structure
    # Introduce some noise/errors: mislabel 15% of points
    noise_mask = np.random.rand(len(true_labels)) < 0.15
    noisy_labels = true_labels.copy()
    noisy_labels[noise_mask] = np.random.randint(1, 4, size=np.sum(noise_mask))
    return noisy_labels

def finetuned_model_segment(pcd):
    """Simulates a fine-tuned model with high accuracy."""
    _, true_labels = generate_synthetic_cloud()
    # Introduce very little noise: mislabel 2% of points
    noise_mask = np.random.rand(len(true_labels)) < 0.02
    noisy_labels = true_labels.copy()
    noisy_labels[noise_mask] = np.random.randint(1, 4, size=np.sum(noise_mask))
    return noisy_labels

# --- Training Simulation Widgets & Logic ---
lr_slider = widgets.FloatLogSlider(value=1e-4, base=10, min=-5, max=-2, step=0.2, description='Learning Rate')
epochs_slider = widgets.IntSlider(value=20, min=5, max=100, step=5, description='Epochs')
train_button = widgets.Button(description="Start Simulated Training", button_style='success')
training_output = widgets.Output()

def simulate_training(b):
    with training_output:
        training_output.clear_output()
        print("Starting training simulation...")
        
        # Generate synthetic loss and accuracy curves
        epochs = epochs_slider.value
        lr = lr_slider.value
        initial_loss = 1.5
        final_loss = 0.1 + np.log10(lr) * -0.05
        loss_curve = initial_loss * np.exp(-np.linspace(0, 3, epochs)) + np.random.rand(epochs) * 0.1 + final_loss
        
        initial_acc = 0.85
        final_acc = 0.98
        acc_curve = final_acc - (final_acc - initial_acc) * np.exp(-np.linspace(0, 3, epochs))
        
        # Plotting
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        ax1.plot(loss_curve, 'r-o', label='Training Loss')
        ax1.set_title('Simulated Loss Curve')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.grid(True)
        
        ax2.plot(acc_curve, 'b-o', label='Validation Accuracy')
        ax2.set_title('Simulated Accuracy Curve')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.grid(True)
        
        plt.tight_layout()
        plt.show()
        
        # Simulate saving the model
        model_path = MODELS_DIR / 'finetuned_sonata_model.json'
        with open(model_path, 'w') as f:
            json.dump({'status': 'trained', 'lr': lr, 'epochs': epochs, 'final_accuracy': acc_curve[-1]},
                      f, indent=4)
        print(f"\nFine-tuned model placeholder saved to: {model_path}")

train_button.on_click(simulate_training)
display(HBox([VBox([lr_slider, epochs_slider]), train_button]))
display(training_output)

## 3. Evaluation & Performance Analysis

Now we evaluate our models. We will:
1.  Generate a confusion matrix for both models to quantify their performance.
2.  Create a side-by-side 3D visualization to qualitatively compare their segmentation results on our test sample.

In [ ]:
def plot_confusion_matrix(true_labels, pred_labels, title):
    """Plots a confusion matrix using seaborn."""
    labels = sorted(np.unique(true_labels))
    cm = confusion_matrix(true_labels, pred_labels, labels=labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

def visualize_comparison(pcd, labels1, labels2, title1, title2):
    """Creates a side-by-side 3D plot for comparing segmentations."""
    points = np.asarray(pcd.points)
    
    def get_colors(labels):
        max_label = labels.max()
        cmap = plt.get_cmap("viridis", max_label + 1)
        return [f'rgb({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)})' for c in cmap(labels)]

    colors1 = get_colors(labels1)
    colors2 = get_colors(labels2)
    
    fig = make_subplots(
        rows=1, cols=2, 
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
        subplot_titles=(title1, title2)
    )
    
    fig.add_trace(go.Scatter3d(x=points[:,0], y=points[:,1], z=points[:,2], mode='markers', marker=dict(size=2, color=colors1)), 1, 1)
    fig.add_trace(go.Scatter3d(x=points[:,0], y=points[:,1], z=points[:,2], mode='markers', marker=dict(size=2, color=colors2)), 1, 2)
    fig.update_layout(title_text="Segmentation Comparison", showlegend=False, height=600)
    fig.show()

# --- Run Evaluation ---
pcd_eval, true_labels_eval = pcd_train, labels_train # Use the same sample for simplicity

# Get predictions
pred_labels_pretrained = pretrained_model_segment(pcd_eval)
pred_labels_finetuned = finetuned_model_segment(pcd_eval)

# Plot confusion matrices
print("--- Performance Evaluation ---")
plot_confusion_matrix(true_labels_eval, pred_labels_pretrained, 'Confusion Matrix (Pre-trained Model)')
plot_confusion_matrix(true_labels_eval, pred_labels_finetuned, 'Confusion Matrix (Fine-tuned Model)')

# Visualize side-by-side
visualize_comparison(pcd_eval, pred_labels_pretrained, pred_labels_finetuned, 'Pre-trained Result', 'Fine-tuned Result')

## 4. Advanced Techniques

This section demonstrates the structure for more advanced MLOps practices.

### Cross-Validation
Below is a function that simulates a k-fold cross-validation loop. In a real scenario, you would split your dataset into k-folds and train/validate the model k times to get a more robust estimate of its performance.

In [ ]:
def simulate_cross_validation(k=5):
    """Simulates a k-fold cross-validation process."""
    print(f"--- Simulating {k}-Fold Cross-Validation ---")
    # In a real scenario, X would be your list of data files and y the labels
    X = np.arange(100) 
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_accuracies = []
    
    for i, (train_index, val_index) in enumerate(kf.split(X)):
        print(f"Fold {i+1}/{k}...")
        print(f"  Train set size: {len(train_index)}, Validation set size: {len(val_index)}")
        # Here you would train your model on the train_set
        # and evaluate on the val_set.
        time.sleep(0.5) # Simulate training time
        fold_accuracy = 0.95 + np.random.uniform(-0.03, 0.03)
        fold_accuracies.append(fold_accuracy)
        print(f"  Validation Accuracy: {fold_accuracy:.4f}")
        
    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)
    print("\nCross-Validation Summary:")
    print(f"  Mean Accuracy: {mean_accuracy:.4f}")
    print(f"  Std Deviation: {std_accuracy:.4f}")

simulate_cross_validation()

## 5. Model Export & Automated Reporting

Finally, we provide a mechanism to generate a complete report summarizing the fine-tuning experiment. Click the button to generate the report.

In [ ]:
report_button = widgets.Button(description="Generate Full Report", button_style='primary')
report_output = widgets.Output()

def generate_full_report(b):
    with report_output:
        report_output.clear_output()
        print("Generating final performance report...")
        
        # --- Report Content ---
        report_str = """
        # Fine-Tuning and Evaluation Report
        
        **Date**: {date}
        
        ## 1. Dataset
        - Source: Synthetic data generated locally.
        - Classes: {classes}
        - Total Points: {num_points}
        
        ## 2. Fine-Tuning Parameters (Simulated)
        - Learning Rate: {lr}
        - Epochs: {epochs}
        
        ## 3. Performance Analysis
        - **Pre-trained Model Accuracy**: {acc_pre:.4f}
        - **Fine-tuned Model Accuracy**: {acc_fine:.4f}
        - **Performance Improvement**: {improvement:+.2%}
        
        --- 
        """.format(
            date=time.strftime("%Y-%m-%d %H:%M:%S"),
            classes=list(np.unique(true_labels_eval)),
            num_points=len(pcd_eval.points),
            lr=lr_slider.value,
            epochs=epochs_slider.value,
            acc_pre=np.mean(pred_labels_pretrained == true_labels_eval),
            acc_fine=np.mean(pred_labels_finetuned == true_labels_eval),
            improvement=(np.mean(pred_labels_finetuned == true_labels_eval) / np.mean(pred_labels_pretrained == true_labels_eval) - 1)
        )
        
        print(report_str)
        
        # Show confusion matrix for the best model
        print("\n**Fine-tuned Model Confusion Matrix:**")
        plot_confusion_matrix(true_labels_eval, pred_labels_finetuned, 'Fine-tuned Model Performance')
        
        # Save report to file
        report_path = REPORTS_DIR / 'evaluation_report.md'
        with open(report_path, 'w') as f:
            f.write(report_str)
        print(f"\nReport saved to {report_path}")

report_button.on_click(generate_full_report)
display(report_button, report_output)